📒 [가이드] 모델 로딩 및 추론 표준 코드

작성자: 박제혁

목적: Hydra 설정과 ModelFactory를 사용하여 모델을 로드하고, 추론을 수행하는 방법을 공유합니다. 각자 담당한 노드(Solver, Critic 등) 구현 시 참고하세요.

In [1]:
import sys
import os
import torch

# 1. 프로젝트 루트 경로 추가 (notebooks/.. -> root)
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.append(project_root)

# 2. 커스텀 모듈 임포트
from src.utils.config_loader import load_config
from src.model.factory import ModelFactory

print(f"✅ 프로젝트 루트가 설정되었습니다: {project_root}")

✅ 프로젝트 루트가 설정되었습니다: /data/ephemeral/home/REPO_Jehyeok


In [2]:
# 설정 로드
cfg = load_config()

# 확인용 출력
print(f"🔹 사용 디바이스: {cfg.system.device}")
print(f"🔹 메인 모델 경로: {cfg.model.main_solver.path}")
print(f"🔹 생성 설정(Generation): {cfg.model.main_solver.generation}")

🔹 사용 디바이스: cuda
🔹 메인 모델 경로: unsloth/Qwen3-32B-bnb-4bit
🔹 생성 설정(Generation): {'max_tokens': 4096, 'temperature': 0.2, 'top_p': 0.9, 'do_sample': True}


In [3]:
# 1. 팩토리 초기화 (모델 설정 전체 전달)
factory = ModelFactory(cfg.model)

# 2. 모델 로드 (시간이 조금 걸릴 수 있습니다)
print("⏳ 모델 로딩 중... (GPU 메모리 확인 필요)")
model, tokenizer = factory.get_model("main_solver")

print("✅ 모델 로딩 완료!")

⏳ 모델 로딩 중... (GPU 메모리 확인 필요)
🔄 [Loader] 모델 로딩 시작: Qwen3-32B-4bit (unsloth/Qwen3-32B-bnb-4bit)
   ↳ ⚡ 양자화 설정 적용 중...


config.json: 0.00B [00:00, ?B/s]

/data/ephemeral/home/REPO_Jehyeok/.venv/lib/python3.11/site-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

✅ [Loader] 로딩 완료!
✅ 모델 로딩 완료!


In [ ]:
# --- 테스트할 질문 ---
question = "2024학년도 수능 수학 22번 문제의 풀이 전략을 간략히 설명해줘."
system_prompt = "당신은 수능 문제를 분석하고 풀이하는 AI 선생님입니다."

# 1. 메시지 구성 (Chat Format)
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

# 2. 프롬프트 변환 (Chat Template 적용)
# 모델마다 학습된 특수 토큰(<|im_start|> 등)을 자동으로 붙여줍니다.
prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(f"📝 실제 모델에 들어가는 입력:\n{prompt_text[:200]}...")

# 3. 토크나이징 & GPU 이동
inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

# 4. 생성 (Generation)
# yaml 파일에 적어둔 top_p, temperature 설정을 여기서 **kwargs로 한 번에 적용합니다.
gen_config = cfg.model.main_solver.generation

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        **gen_config  # config/model/qwen3_32b.yaml 의 설정 적용
    )

# 5. 디코딩 (결과 확인)
# 입력 프롬프트 길이를 제외하고 새로 생성된 부분만 잘라냅니다.
input_length = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][input_length:]
answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("\n" + "="*30)
print(f"🤖 모델 답변:\n{answer}")
print("="*30)